### Simple GenAI App Using LangChain And OpenAI Model

In [ ]:
# LangChain imports
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Community integrations
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS

# OpenAI integrations
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_openai import ChatOpenAI

# Environment variables
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# Scraping the target website for the document
loader = WebBaseLoader("https://docs.langchain.com/langsmith/observability-llm-tutorial")
docs = loader.load()
docs

c:\Users\viren\anaconda3\envs\langchain_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
# Splitting the document into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)
documents

In [ ]:
# Converting the chunks into vectors and storing it in the vector store
embeddings = OpenAIEmbeddings()
vector_store = FAISS.from_documents(documents, embeddings)

In [ ]:
# Storing the vector store locally and loading from it
vector_store.save_local("faiss_index")
vector_store = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

In [ ]:
# Converting the vector store into retriever
retriever = vector_store.as_retriever()

In [ ]:
# Initializing the openai llm model
llm = ChatOpenAI(model='gpt-4o')
print(llm)

In [ ]:
# Creating the custom prompt template
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system", 
            """
            You are an expert AI assistant. Answer the question based ONLY on the provided context. 
            Context: {context}
            """
        ),
        ("user", "{input}")
    ]
)

prompt

In [ ]:
# Initializing the output parser  
output_parser = StrOutputParser()

In [ ]:
# Creating the lcel chain
chain = ( {"context": retriever, "input": RunnablePassthrough()} | prompt | llm | output_parser)

In [ ]:
# Querying from the llm
input_query = "Summarize the document"
response = chain.invoke(input_query)
response